<a href="https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/simulation/Specification_debt_simulation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#This is code to manage dependencies if the notebook is executed in the google colab cloud service
if 'google.colab' in str(get_ipython()):
  import os
  os.system('apt -qqq install graphviz')
  os.system('pip -qqq install ModelFlowIb')

In [2]:
import pandas as pd
import re

from modelclass import model
import modelmanipulation as mp
from modelmanipulation import doable, explode

from modelpattern import list_extract
import modeljupytermagic

In [3]:
# just a helper for colab reference 

from modelhelp import colab_link
colab_link('Specification_debt_simulation_model',badge=True,render=0)

<a href="https://colab.research.google.com/github/IbHansen/wb-debt-simulation/blob/main/simulation/Specification_debt_simulation_model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# About lists

**lists** and **sublists** are central to making ModelFlow scalable and expressive.  
They allow the modeler to define **groups of entities** (such as bonds, banks, portfolios, sectors, or countries) and to attach **attributes** or **hierarchies** to those entities.  
A list definition typically takes the following form

$list\: \underbrace{issued}_{List name} = \underbrace{issued}_{sub listname} : issued\_2025 * issued\_2050 / \\
\underbrace{issued\_year}_{sub listname} : 2025 * 2050$

Please note:
 - the list name and the first sub list name should be the same
 - case don't matter
 - a * combined with identifiers ending in digits will construct multiple list content

  `issued_2025 * issued_2050` becomes `issued_2025 issued_2026 … issued_2050`


This enables equations to be written once as templates and then **expanded automatically** across all relevant list members.  

Sublists can be used to define logical relationships between entities — for example, linking each bond to its home maturity, or to define the year for each issurance year.  
These structures make it possible to organize models systematically and to maintain consistency across entities that share the same behavioral equations.  

This approach eliminates redundancy and manual repetition: instead of writing dozens or hundreds of similar equations, the modeler defines a **single equation pattern**, and ModelFlow expands it dynamically over the members of a list.  

In [4]:
%%Makemymodel hist segment=listbonds
## List definitions


>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000         /
>>              10_hist_usd        10      0       usd         0        1    2         20        

>list currency = currency : dom usd    

## List definitions


```text
>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000         /
>>              10_hist_usd        10      0       usd         0        1    2         20        
```

```text
>list currency = currency : dom usd    
```

In [5]:
port_replacements=[('{issued_year}','2025'),
                   ('__bond','__{bond}'),
                   ('fx_rate','dom_{currency}')
                  ]

In [6]:
%%Makemymodel hist segment=logical replacements=port_replacements Nshow render=0 Nrender_list=False

## Define logical variables
The folowing variables are defined for each bond and each issuing year. They are **logical variables** that is they can be either 0.0 or 1.0.

The terms enclosed in `{}` are taken from the relevant sublists. the  `current_year` are taken from the input dataset.

> doable logical_amortizing_years__bond =  ( {issued_year}+{grace}) < current_year <= ({issued_year} + {maturity})


## List definitions


```text
>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000         /
>>              10_hist_usd        10      0       usd         0        1    2         20        
```

```text
>list currency = currency : dom usd    
```

## Define logical variables
The folowing variables are defined for each bond and each issuing year. They are **logical variables** that is they can be either 0.0 or 1.0.

The terms enclosed in `{}` are taken from the relevant sublists. the  `current_year` are taken from the input dataset.

```text
> doable logical_amortizing_years__bond =  ( {issued_year}+{grace}) < current_year <= ({issued_year} + {maturity})
```

In [7]:
%%Makemymodel hist segment=stock replacements=port_replacements  Nshow Nrender=0 Nrender_list=False

## Currency amortizing and stock laws of motion
> doable amortizing_share_pr_time__bond =  logical_amortizing_years__bond / ({maturity}-{grace})
### Amortizing in fx 
> doable  amortizing_in_currency__bond = {start_in_currency} * amortizing_share_pr_time__bond

### End of year stock in fx
> doable stock_ultimo_in_currency__bond = stock_primo_in_currency__bond-amortizing_in_currency__bond

### Start of year stock in fx 
> doable stock_primo_in_currency__bond = (current_year=={issued_year}+1)*{start_in_currency} +stock_ultimo_in_currency__bond(-1)




## List definitions


```text
>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000         /
>>              10_hist_usd        10      0       usd         0        1    2         20        
```

```text
>list currency = currency : dom usd    
```

## Currency amortizing and stock laws of motion
```text
> doable amortizing_share_pr_time__bond =  logical_amortizing_years__bond / ({maturity}-{grace})
```
### Amortizing in fx 
```text
> doable  amortizing_in_currency__bond = {start_in_currency} * amortizing_share_pr_time__bond
```

### End of year stock in fx
```text
> doable stock_ultimo_in_currency__bond = stock_primo_in_currency__bond-amortizing_in_currency__bond
```

### Start of year stock in fx 
```text
> doable stock_primo_in_currency__bond = (current_year=={issued_year}+1)*{start_in_currency} +stock_ultimo_in_currency__bond(-1)
```



In [8]:
%%Makemymodel hist segment=interest_rate replacements=port_replacements  nshow Nrender=0 Nrender_list=False

## Currency interest rate payment 
> doable interest_in_currency__bond = stock_primo_in_currency__bond * {rate}/100 

## List definitions


```text
>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000         /
>>              10_hist_usd        10      0       usd         0        1    2         20        
```

```text
>list currency = currency : dom usd    
```

## Currency interest rate payment 
```text
> doable interest_in_currency__bond = stock_primo_in_currency__bond * {rate}/100 
```

In [9]:
%%Makemymodel hist segment=sums replacements=port_replacements  nshow render=0 Nrender_list=False
## Domestic amortizing, stocks and interest payments 
### Amortizing in domestic currency 

> doable <sum=hist>  amortizing__bond = amortizing_in_currency__bond * fx_rate

### Stock in domestic currency 
> doable <sum=hist> stock_primo__bond  = stock_primo_in_currency__bond  * fx_rate
> doable <sum=hist> stock_ultimo__bond = stock_ultimo_in_currency__bond * fx_rate

### Interest rate payment

> doable <sum=hist>  interest_payments__bond    = interest_in_currency__bond  * fx_rate

## List definitions


```text
>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000         /
>>              10_hist_usd        10      0       usd         0        1    2         20        
```

```text
>list currency = currency : dom usd    
```
## Domestic amortizing, stocks and interest payments 
### Amortizing in domestic currency 

```text
> doable <sum=hist>  amortizing__bond = amortizing_in_currency__bond * fx_rate
```

### Stock in domestic currency 
```text
> doable <sum=hist> stock_primo__bond  = stock_primo_in_currency__bond  * fx_rate
> doable <sum=hist> stock_ultimo__bond = stock_ultimo_in_currency__bond * fx_rate
```

### Interest rate payment

```text
> doable <sum=hist>  interest_payments__bond    = interest_in_currency__bond  * fx_rate
```

In [10]:
%Makemymodel hist replacements=port_replacements
mhist = hist.mmodel


## Define logical variables
The folowing variables are defined for each bond and each issuing year. They are **logical variables** that is they can be either 0.0 or 1.0.

The terms enclosed in `{}` are taken from the relevant sublists. the  `current_year` are taken from the input dataset.

```text
> doable logical_amortizing_years__bond =  ( {issued_year}+{grace}) < current_year <= ({issued_year} + {maturity})
```


## Currency amortizing and stock laws of motion
```text
> doable amortizing_share_pr_time__bond =  logical_amortizing_years__bond / ({maturity}-{grace})
```
### Amortizing in fx 
```text
> doable  amortizing_in_currency__bond = {start_in_currency} * amortizing_share_pr_time__bond
```

### End of year stock in fx
```text
> doable stock_ultimo_in_currency__bond = stock_primo_in_currency__bond-amortizing_in_currency__bond
```

### Start of year stock in fx 
```text
> doable stock_primo_in_currency__bond = (current_year=={issued_year}+1)*{start_in_currency} +stock_ultimo_in_currency__bond(-1)
```




## Currency interest rate payment 
```text
> doable interest_in_currency__bond = stock_primo_in_currency__bond * {rate}/100 
```

## Domestic amortizing, stocks and interest payments 
### Amortizing in domestic currency 

```text
> doable <sum=hist>  amortizing__bond = amortizing_in_currency__bond * fx_rate
```

### Stock in domestic currency 
```text
> doable <sum=hist> stock_primo__bond  = stock_primo_in_currency__bond  * fx_rate
> doable <sum=hist> stock_ultimo__bond = stock_ultimo_in_currency__bond * fx_rate
```

### Interest rate payment

```text
> doable <sum=hist>  interest_payments__bond    = interest_in_currency__bond  * fx_rate
```

## List definitions


```text
>tlist bond =        bond  maturity  grace  currency  domestic  foreign    rate  start_in_currency /
>>              10_hist_dom        10      0       dom         1        0    3        1000         /
>>              10_hist_usd        10      0       usd         0        1    2         20        
```

```text
>list currency = currency : dom usd    
```

In [11]:
years = [y for y in range(2025,2050,1)]
dfstart = pd.DataFrame(years,index=years,columns=['CURRENT_YEAR'])
#dfstart

In [12]:
mhist.exogene

{'CURRENT_YEAR', 'DOM_DOM', 'DOM_USD'}

In [13]:
df = dfstart.upd('''
dom_dom = 1
dom_usd = 10
''')
df.T


,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,...,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049
CURRENT_YEAR,2025.0,2026.0,2027.0,2028.0,2029.0,2030.0,2031.0,2032.0,2033.0,2034.0,...,2040.0,2041.0,2042.0,2043.0,2044.0,2045.0,2046.0,2047.0,2048.0,2049.0
DOM_DOM,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
DOM_USD,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,...,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0,10.0


In [14]:
res = mhist(df,2026,2050,silent=1)

In [15]:
with mhist.set_smpl(2025,2050): 
    display(mhist['stock_primo_in_currency__* stock*HIST interest*all amort* amort*HIST interest*hist'].df)

,STOCK_PRIMO_IN_CURRENCY__10_HIST_DOM,STOCK_PRIMO_IN_CURRENCY__10_HIST_USD,STOCK_PRIMO__HIST,STOCK_ULTIMO__HIST,AMORTIZING_IN_CURRENCY__10_HIST_DOM,AMORTIZING_IN_CURRENCY__10_HIST_USD,AMORTIZING_SHARE_PR_TIME__10_HIST_DOM,AMORTIZING_SHARE_PR_TIME__10_HIST_USD,AMORTIZING__10_HIST_DOM,AMORTIZING__10_HIST_USD,AMORTIZING__HIST,AMORTIZING__HIST,INTEREST_PAYMENTS__HIST
2025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2026,1000.0,20.0,1200.0,1080.0,100.0,2.0,0.1,0.1,100.0,20.0,120.0,120.0,34.0
2027,900.0,18.0,1080.0,960.0,100.0,2.0,0.1,0.1,100.0,20.0,120.0,120.0,30.6
2028,800.0,16.0,960.0,840.0,100.0,2.0,0.1,0.1,100.0,20.0,120.0,120.0,27.2
2029,700.0,14.0,840.0,720.0,100.0,2.0,0.1,0.1,100.0,20.0,120.0,120.0,23.8
2030,600.0,12.0,720.0,600.0,100.0,2.0,0.1,0.1,100.0,20.0,120.0,120.0,20.4
2031,500.0,10.0,600.0,480.0,100.0,2.0,0.1,0.1,100.0,20.0,120.0,120.0,17.0
2032,400.0,8.0,480.0,360.0,100.0,2.0,0.1,0.1,100.0,20.0,120.0,120.0,13.6
2033,300.0,6.0,360.0,240.0,100.0,2.0,0.1,0.1,100.0,20.0,120.0,120.0,10.2
2034,200.0,4.0,240.0,120.0,100.0,2.0,0.1,0.1,100.0,20.0,120.0,120.0,6.8


In [16]:
mhist['amortizing*'].df

,AMORTIZING_IN_CURRENCY__10_HIST_DOM,AMORTIZING_IN_CURRENCY__10_HIST_USD,AMORTIZING_SHARE_PR_TIME__10_HIST_DOM,AMORTIZING_SHARE_PR_TIME__10_HIST_USD,AMORTIZING__10_HIST_DOM,AMORTIZING__10_HIST_USD,AMORTIZING__HIST
2026,100.0,2.0,0.1,0.1,100.0,20.0,120.0
2027,100.0,2.0,0.1,0.1,100.0,20.0,120.0
2028,100.0,2.0,0.1,0.1,100.0,20.0,120.0
2029,100.0,2.0,0.1,0.1,100.0,20.0,120.0
2030,100.0,2.0,0.1,0.1,100.0,20.0,120.0
2031,100.0,2.0,0.1,0.1,100.0,20.0,120.0
2032,100.0,2.0,0.1,0.1,100.0,20.0,120.0
2033,100.0,2.0,0.1,0.1,100.0,20.0,120.0
2034,100.0,2.0,0.1,0.1,100.0,20.0,120.0
2035,100.0,2.0,0.1,0.1,100.0,20.0,120.0


In [17]:
mhist.modeldump('model/hist')

In [18]:
mhist.drawmodel(svg=True,browser=True)

In [19]:
hist.report().save(open_file=True)

✔ Report saved → C:\wb debt simulation\wb-debt-simulation\simulation_2\html\model-report_report.html


'html\\model-report_report.html'